### Structured Output 

Models can be requested to provide their response in a format matching a given schema . This is useful for ensuring the output can be easily parsed and used in subsequent processing . LangChain supports multiple schema types and mathods for enforcing structured output

### Pydantic 

Pydantic models provide the richest feature set with fields validation , descriptions , and nested structures .

In [1]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F4F7B5ED50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F4F7CA7550>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The movie was released this year")
    director:str=Field(description="The director of this movie")
    rating:float=Field(description="The movies rating out of 10")

In [3]:
structure_model = model.with_structured_output(Movie)
structure_model

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F4F7B5ED50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F4F7CA7550>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The ti

In [4]:
structure_model.invoke("Provide Details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output along side parsed structure 

In [5]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="The movie was released this year")
    director:str=Field(...,description="The director of this movie")
    rating:float=Field(...,description="The movies rating out of 10")

structure_model = model.with_structured_output(Movie , include_raw=True)

res = structure_model.invoke("Provide Details about the movie inception")
res

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants details about the movie Inception. We can use the function Movie to get details: director, rating, title, year. Provide answer.', 'tool_calls': [{'id': 'fc_879fe72a-7358-4221-8700-b2a98802878e', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 157, 'total_tokens': 240, 'completion_time': 0.177987135, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.006702909, 'prompt_tokens_details': None, 'queue_time': 0.349503151, 'total_time': 0.184690044}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8ea50c0161', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a08131-576a-7d02-af37-cd4f267b7b54-0', tool_calls=[{'name': 'Movie', 'args':

### Nested Structure 

In [ ]:
from pydantic import BaseModel

class Actor(BaseModel):
    name: str 
    role: str 


class MovieDetails(BaseModel):
    title: str 
    year: int 
    cast: list[Actor]
    generes: list[str]
    bugdet: float | None = Field(None , description="Budget in millions USD")

structure_model = model.with_structured_output(MovieDetails)
res = structure_model.invoke("Provide Details about the movie inception")
res

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles')], generes=['Science Fiction', 'Action', 'Thriller', 'Heist'], bugdet=None)

### TypedDict

provides a similar alternative using Pythons build in typing , ideal when you dont need runtime validation

In [8]:
from typing_extensions import TypedDict , Annotated

class MovieDict(TypedDict):
    title : Annotated[str , ... , "tHE TITLE OF THE MOVIE"]
    year :  Annotated[int,...,"Year of release"]
    director : Annotated[str,...,"The director of the movie"]
    rating : Annotated[float , ... , "Rating out of 10"]


strucute_model = model.with_structured_output(MovieDict)
res = strucute_model.invoke("Provide Details about the movie inception")
res

{'director': 'Christopher Nolan',
 'rating': 8.8,
 'title': 'Inception',
 'year': 2010}

In [12]:
class Actor(TypedDict):
    name: str 
    role: str 


class MovieDetails(TypedDict):
    title: str 
    year: int 
    cast: list[Actor]
    generes: list[str]
    budget: float | None = Field(None , description="Budget in millions USD")

structure_model = model.with_structured_output(MovieDetails)
res = structure_model.invoke("Provide Details about the movie Avengers")
res

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'generes': ['Action', 'Adventure', 'Sci-Fi', 'Superhero'],
 'title': 'The Avengers',
 'year': 2012}

### DataClasses 

A data class is a class typically containing mainly data , although there are not really any restrictions . You create it using the @dataclass decorator

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

In [13]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")
    
agent = create_agent(
    model="google_genai:gemini-2.5-flash",  # or "google_genai:gemini-1.5-flash"
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='d5643f1e-1a1d-4e4d-bdac-e8b2efd02a13'),
  AIMessage(content='{\n"name": "John Doe",\n"email": "john@example.com",\n"phone": "(555) 123-4567"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0814b-3c7c-7ab2-abb5-a67f1a83e432-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 129, 'total_tokens': 158, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 88}})],
 'structured_response': ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')}

In [14]:
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [15]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="google_genai:gemini-2.5-flash",  # or "google_genai:gemini-1.5-flash"
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [16]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person



agent = create_agent(
    model="google_genai:gemini-2.5-flash",  # or "google_genai:gemini-1.5-flash"
    response_format=ContactInfo
)


result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')